# 04 — Depth Precomputation

Runs MiDaS DPT_Hybrid depth estimation offline for all KITTI and CityPersons images.

**This is a one-time offline step. MiDaS is NEVER called during training.**

**What this notebook does:**
1. Precomputes depth maps and confidence maps for KITTI (train/val/test)
2. Precomputes for CityPersons (train/val)
3. Verifies output `.npy` files are the correct shape
4. Saves 20 side-by-side RGB/depth/confidence visualisations
5. Runs the depth consistency augmentation test (before/after with mask overlay)

**Gate: this notebook MUST complete before running notebooks 05–08.**

**Output directories:**
- `data/kitti/depth_hybrid/training/image_2/{id}.npy`
- `data/kitti/depth_conf/training/image_2/{id}.npy`
- `data/citypersons/depth_hybrid/{split}/{city}/{stem}.npy`
- `data/citypersons/depth_conf/{split}/{city}/{stem}.npy`

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.depth import MiDaSDepthEstimator
from src.config import load_config

ROOT = Path('..').resolve()
DATA_ROOT = ROOT / 'data'
FIGURES_DIR = ROOT / 'results' / 'figures' / 'depth_consistency_check'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'Data root: {DATA_ROOT}')

## 1. KITTI Depth Precomputation

In [ ]:
kitti_image_dir = DATA_ROOT / 'kitti' / 'data_object_image_2' / 'training' / 'image_2'
kitti_depth_dir = DATA_ROOT / 'kitti' / 'depth_hybrid' / 'training' / 'image_2'

if kitti_image_dir.exists():
    estimator = MiDaSDepthEstimator(device=device)

    print('Precomputing KITTI depth maps...')
    estimator.precompute_dataset(
        image_dir=kitti_image_dir,
        output_dir=kitti_depth_dir,
        batch_size=8,
        vis_every=200,
        vis_dir=FIGURES_DIR / 'kitti',
    )
    print(f'KITTI depth maps saved to: {kitti_depth_dir}')
else:
    print(f'[SKIP] KITTI images not found at: {kitti_image_dir}')

## 2. CityPersons Depth Precomputation

In [ ]:
cp_root = DATA_ROOT / 'citypersons'
cp_img_root = cp_root / 'leftImg8bit_trainvaltest'

if cp_img_root.exists():
    if 'estimator' not in dir():
        estimator = MiDaSDepthEstimator(device=device)

    for split in ['train', 'val']:
        split_img_dir = cp_img_root / split
        if not split_img_dir.exists():
            print(f'[SKIP] CityPersons {split} images not found')
            continue

        print(f'Precomputing CityPersons {split} depth maps...')
        # Process per-city to maintain the depth_hybrid/{split}/{city}/{stem}.npy structure
        for city_dir in sorted(split_img_dir.iterdir()):
            if not city_dir.is_dir():
                continue
            out_dir = cp_root / 'depth_hybrid' / split / city_dir.name
            estimator.precompute_dataset(
                image_dir=city_dir,
                output_dir=out_dir,
                batch_size=8,
                vis_every=500,
                vis_dir=FIGURES_DIR / 'citypersons',
            )
        print(f'CityPersons {split} depth maps complete.')
else:
    print(f'[SKIP] CityPersons images not found at: {cp_img_root}')

## 3. Verification — shape and value checks

In [ ]:
def verify_depth_files(depth_dir, dataset_name, n_check=20):
    """Spot-check depth .npy files for correct shape and value range."""
    depth_files = sorted(Path(depth_dir).rglob('*.npy'))
    depth_files = [f for f in depth_files if '_conf' not in f.name]

    if not depth_files:
        print(f'[SKIP] No depth files found at {depth_dir}')
        return

    print(f'\n{dataset_name}: {len(depth_files)} depth files found.')
    failures = []
    for f in depth_files[:n_check]:
        arr = np.load(f)
        conf_path = f.parent / (f.stem + '_conf.npy')
        if arr.ndim != 2:
            failures.append(f'{f.name}: expected 2D, got {arr.shape}')
        if arr.dtype != np.float32:
            failures.append(f'{f.name}: expected float32, got {arr.dtype}')
        if arr.min() < 0 or arr.max() > 1.01:
            failures.append(f'{f.name}: values out of [0,1] range: [{arr.min():.3f}, {arr.max():.3f}]')
        if not conf_path.exists():
            failures.append(f'Missing conf file: {conf_path.name}')

    if failures:
        for msg in failures:
            print(f'  [FAIL] {msg}')
        raise AssertionError(f'{dataset_name}: {len(failures)} verification failures.')
    else:
        sample = np.load(depth_files[0])
        print(f'  [OK] Shape: {sample.shape}, dtype: {sample.dtype}, range: [{sample.min():.3f}, {sample.max():.3f}]')
        print(f'  [OK] {min(n_check, len(depth_files))} files checked — all valid.')

if kitti_depth_dir.exists():
    verify_depth_files(kitti_depth_dir, 'KITTI')

cp_depth_root = DATA_ROOT / 'citypersons' / 'depth_hybrid'
if cp_depth_root.exists():
    verify_depth_files(cp_depth_root, 'CityPersons')

## 4. Depth Consistency Augmentation Test

Verifies the critical invariant: when augmentation masks a region, `depth[masked]=0.5`, `depth_conf[masked]=0.0`, `depth_mask[masked]=False`.

In [ ]:
import torch
from src.augmentation import LabelAwareCutout, RealOccluderAugmentation, LabelAwareGridMask

N_EXAMPLES = 20
fig, axes = plt.subplots(N_EXAMPLES, 4, figsize=(16, 4 * N_EXAMPLES))
if N_EXAMPLES == 1:
    axes = axes[None, :]

cutout = LabelAwareCutout(p=1.0)
rng = torch.Generator().manual_seed(42)

all_passed = True
for idx in range(N_EXAMPLES):
    img   = torch.rand(3, 640, 640)
    depth = torch.rand(1, 640, 640)
    conf  = torch.rand(1, 640, 640)
    dmask = torch.ones(1, 640, 640, dtype=torch.bool)
    boxes = torch.tensor([
        [0.1 + idx*0.01 % 0.3, 0.1, 0.5 + idx*0.01 % 0.3, 0.5]
    ])
    occ_lvls = [0]

    result = cutout(img, depth, conf, dmask, boxes, occ_lvls)

    # Verify invariants
    mask = result.mask_applied
    if mask.any():
        depth_vals = result.depth[0][mask]
        conf_vals  = result.depth_conf[0][mask]
        dmask_vals = result.depth_mask[0][mask]

        depth_ok = (depth_vals == 0.5).all().item()
        conf_ok  = (conf_vals  == 0.0).all().item()
        mask_ok  = (~dmask_vals).all().item()

        if not (depth_ok and conf_ok and mask_ok):
            all_passed = False
            print(f'[FAIL] Example {idx}: depth_ok={depth_ok}, conf_ok={conf_ok}, mask_ok={mask_ok}')

    # Plot: original RGB | augmented RGB | depth before | depth after (masked region visible)
    axes[idx, 0].imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
    axes[idx, 0].set_title('Original RGB' if idx == 0 else '')
    axes[idx, 0].axis('off')

    axes[idx, 1].imshow(result.image.permute(1, 2, 0).clamp(0, 1).numpy())
    axes[idx, 1].set_title('Augmented RGB' if idx == 0 else '')
    axes[idx, 1].axis('off')

    axes[idx, 2].imshow(depth[0].numpy(), cmap='plasma', vmin=0, vmax=1)
    axes[idx, 2].set_title('Depth (before)' if idx == 0 else '')
    axes[idx, 2].axis('off')

    aug_depth_display = result.depth[0].numpy().copy()
    overlay = axes[idx, 3].imshow(aug_depth_display, cmap='plasma', vmin=0, vmax=1)
    if mask.any():
        # Highlight masked region in red
        mask_overlay = np.zeros((*aug_depth_display.shape, 4))
        mask_overlay[mask.numpy()] = [1, 0, 0, 0.4]
        axes[idx, 3].imshow(mask_overlay)
    axes[idx, 3].set_title('Depth after (red=masked, val=0.5)' if idx == 0 else '')
    axes[idx, 3].axis('off')

plt.suptitle('Depth Consistency Check: fill=0.5, conf=0, mask=False', fontsize=13, y=1.001)
plt.tight_layout()
out = FIGURES_DIR / 'depth_consistency_check.png'
plt.savefig(out, dpi=100, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

if all_passed:
    print('\n[PASS] All depth consistency checks passed. Safe to proceed with augmentation training.')
else:
    raise AssertionError('Depth consistency check FAILED — do not proceed to notebook 05.')

## 5. Side-by-side RGB / Depth / Confidence visualisations (20 examples)

In [ ]:
from PIL import Image

kitti_img_dir = DATA_ROOT / 'kitti' / 'data_object_image_2' / 'training' / 'image_2'

if kitti_depth_dir.exists() and kitti_img_dir.exists():
    depth_files = sorted(kitti_depth_dir.glob('*.npy'))[:20]
    fig, axes = plt.subplots(len(depth_files), 3, figsize=(12, 4 * len(depth_files)))
    if len(depth_files) == 1:
        axes = axes[None, :]

    for i, df in enumerate(depth_files):
        img_path = kitti_img_dir / (df.stem + '.png')
        conf_path = df.parent / (df.stem + '_conf.npy')

        depth_arr = np.load(df)
        conf_arr  = np.load(conf_path) if conf_path.exists() else np.zeros_like(depth_arr)
        img_arr   = np.array(Image.open(img_path)) if img_path.exists() else np.zeros((375, 1242, 3), np.uint8)

        axes[i, 0].imshow(img_arr)
        axes[i, 0].set_title(f'RGB: {df.stem}' if i == 0 else df.stem)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(depth_arr, cmap='plasma', vmin=0, vmax=1)
        axes[i, 1].set_title('Depth' if i == 0 else '')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(conf_arr, cmap='viridis', vmin=0, vmax=1)
        axes[i, 2].set_title('Confidence' if i == 0 else '')
        axes[i, 2].axis('off')

    plt.suptitle('KITTI: RGB | Depth | Confidence (20 examples)', fontsize=12)
    plt.tight_layout()
    out = FIGURES_DIR / 'kitti_depth_examples.png'
    plt.savefig(out, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')
else:
    print('[SKIP] KITTI depth or image files not found.')

print('\nNotebook 04 complete. Depth precomputation gate passed.')
print('Proceed to notebook 05_augmentation.')